# 06 — So sánh dev và khóa cấu hình KNN cuối

Đọc kết quả của notebook 02, 03, 04, 05A và 05B. Chọn cấu hình trên dev bằng số chưa làm tròn: Macro-F1 cao nhất → MAE thấp hơn khi hòa → thứ tự ứng viên khi vẫn hòa.

Notebook chỉ tổng hợp kết quả và ghi cấu hình khóa, không fit lại KNN/PCA hoặc đọc test. Sau khi chạy bước test, không quay lại chọn tham số theo lỗi test. Test dự án đã được xem trong khảo sát trước, nên lần đánh giá mới không được gọi là test hoàn toàn chưa xem.

In [1]:
import json
import hashlib
import platform
import sklearn
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

PROJECT_ROOT = Path(r"C:\CS231")
RESULTS_ROOT = PROJECT_ROOT / "src" / "KNN" / "results_224"
output_dir = RESULTS_ROOT / "selection"
output_dir.mkdir(parents=True, exist_ok=True)

candidates = [
    ("HSV12", "hsv12", "hsv12", "hsv12/features"),
    ("HOG", "hog", "hog", "hog"),
    ("HOG_PCA", "hog_pca", "hog_pca", "hog_pca"),
    ("HSV12_HOG", "hsv12_hog", "fusion", "hsv12_hog"),
    ("HSV12_HOG_PCA", "hsv12_hog_pca", "fusion", "hsv12_hog_pca"),
]

## 1. Tổng hợp 100 cấu hình

Mỗi biến thể có 20 cấu hình: k=1..10, uniform/distance, Euclidean. Thứ tự biến thể đúng theo lộ trình; trong mỗi biến thể, k tăng dần và uniform trước distance. Các file lưới đã được sắp theo điểm, nên phải tạo lại thứ tự thử khi phá hòa.

In [2]:
grids = []
source_files = []

for feature_order, (feature_id, folder, prefix, feature_folder) in enumerate(candidates):
    grid_file = RESULTS_ROOT / folder / f"{prefix}_knn_grid_dev.csv"
    grid = pd.read_csv(grid_file)
    grid["feature_id"] = feature_id
    grid["feature_order"] = feature_order
    grid["weight_order"] = grid["weights"].map({"uniform": 0, "distance": 1})
    grid["feature_folder"] = feature_folder
    grid["result_folder"] = folder
    grids.append(grid)
    source_files.append(grid_file)

all_results = pd.concat(grids, ignore_index=True).sort_values(
    ["macro_f1", "mae_vnd", "feature_order", "k", "weight_order"],
    ascending=[False, True, True, True, True]
).reset_index(drop=True)
all_results.to_csv(output_dir / "all_100_configurations_dev.csv", index=False)

print("Số cấu hình:", len(all_results))
display(all_results.groupby("feature_id", sort=False).size().rename("configurations"))

Số cấu hình: 100


feature_id
HSV12            20
HSV12_HOG        20
HSV12_HOG_PCA    20
HOG_PCA          20
HOG              20
Name: configurations, dtype: int64

## 2. So sánh năm biến thể

Giữ ba họ đặc trưng chính và hiển thị đầy đủ biến thể PCA để minh bạch phạm vi đã thử. Bảng đầu chọn tốt nhất mỗi biến thể; bảng sau cố định k=1, uniform, Euclidean để so biểu diễn khi giữ bộ phân loại.

In [3]:
best_by_feature = all_results.drop_duplicates("feature_id", keep="first")
reference = all_results.loc[
    (all_results["k"] == 1) & (all_results["weights"] == "uniform")
].sort_values("feature_order")

columns = ["feature_id", "k", "weights", "metric", "accuracy", "macro_f1", "mae_vnd"]
print("Cấu hình tốt nhất mỗi biến thể:")
display(best_by_feature[columns])
print("Cấu hình tham chiếu cố định:")
display(reference[columns])

best_by_feature[columns].to_csv(output_dir / "best_by_feature_dev.csv", index=False)
reference[columns].to_csv(output_dir / "reference_dev.csv", index=False)

Cấu hình tốt nhất mỗi biến thể:


,feature_id,k,weights,metric,accuracy,macro_f1,mae_vnd
0,HSV12,1,uniform,euclidean,0.796036,0.798307,46585.677749
20,HSV12_HOG,6,distance,euclidean,0.729540,0.731995,62071.611253
30,HSV12_HOG_PCA,6,distance,euclidean,0.710997,0.713416,66086.956522
60,HOG_PCA,5,distance,euclidean,0.443734,0.437934,106898.976982
79,HOG,1,uniform,euclidean,0.390665,0.390748,116234.015345


Cấu hình tham chiếu cố định:


,feature_id,k,weights,metric,accuracy,macro_f1,mae_vnd
0,HSV12,1,uniform,euclidean,0.796036,0.798307,46585.677749
79,HOG,1,uniform,euclidean,0.390665,0.390748,116234.015345
61,HOG_PCA,1,uniform,euclidean,0.434783,0.431541,110249.360614
23,HSV12_HOG,1,uniform,euclidean,0.726343,0.726608,58740.409207
44,HSV12_HOG_PCA,1,uniform,euclidean,0.699488,0.701336,66208.439898


## 3. Chọn một cấu hình cuối

Chọn dòng đầu sau khi sắp xếp, không chọn theo kỳ vọng phương án kết hợp phải thắng. Đường dẫn feature và nhãn cho phép bước test dùng đúng train đã có, không tự gộp train+dev.

In [4]:
best = all_results.iloc[0]
feature_dir = RESULTS_ROOT / best["feature_folder"]
X_train = np.load(feature_dir / "X_train.npy", mmap_mode="r")
X_dev = np.load(feature_dir / "X_dev.npy", mmap_mode="r")
y_train = np.load(feature_dir / "y_train.npy")
y_dev = np.load(feature_dir / "y_dev.npy")

print("Phương án được chọn:", best["feature_id"])
print("k:", int(best["k"]), "| weights:", best["weights"], "| metric:", best["metric"])
print("Train/dev feature:", X_train.shape, X_dev.shape)
print("Train/dev nhãn:", y_train.shape, y_dev.shape)
display(pd.DataFrame([best[columns].to_dict()]))

Phương án được chọn: HSV12
k: 1 | weights: uniform | metric: euclidean
Train/dev feature: (12494, 1728) (1564, 1728)
Train/dev nhãn: (12494,) (1564,)


,feature_id,k,weights,metric,accuracy,macro_f1,mae_vnd
0,HSV12,1,uniform,euclidean,0.796036,0.798307,46585.677749


## 4. Ghi cấu hình khóa và dấu vân tay các file đầu vào

SHA-256 giúp bước test phát hiện file cấu hình, lưới hoặc feature đã thay đổi sau khi khóa. Hash không chứng minh dữ liệu không có leakage. Các đường dẫn ghi tương đối theo repository để thành viên khác dùng lại được.

Lưu toàn bộ quy tắc tiền xử lý của bước 1, đối tượng PCA nếu phương án chọn cần, cùng nhãn và thứ tự ảnh. Chưa tạo mô hình hoặc kết quả test.

In [5]:
def file_hash(path):
    with path.open("rb") as file:
        return hashlib.file_digest(file, "sha256").hexdigest()


experiment_file = RESULTS_ROOT / "inputs" / "experiment_config.json"
experiment_config = json.loads(experiment_file.read_text(encoding="utf-8"))
files_to_check = source_files + [experiment_file]
for filename in ["X_train.npy", "X_dev.npy", "y_train.npy", "y_dev.npy", "train_manifest.csv", "dev_manifest.csv"]:
    files_to_check.append(feature_dir / filename)

uses_pca = "PCA" in best["feature_id"]
pca_file = RESULTS_ROOT / "hog_pca" / "pca.pkl"
if uses_pca:
    files_to_check.append(pca_file)

fingerprints = {
    path.relative_to(PROJECT_ROOT).as_posix(): file_hash(path)
    for path in files_to_check
}

locked_config = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "feature_id": str(best["feature_id"]),
    "feature_dim": int(X_train.shape[1]),
    "feature_dtype": str(X_train.dtype),
    "feature_folder": feature_dir.relative_to(PROJECT_ROOT).as_posix(),
    "result_folder": (RESULTS_ROOT / best["result_folder"]).relative_to(PROJECT_ROOT).as_posix(),
    "classes_vnd": experiment_config["classes_vnd"],
    "preprocessing": experiment_config["preprocessing"],
    "experiment_protocol": experiment_config,
    "uses_pca": bool(uses_pca),
    "pca_file": pca_file.relative_to(PROJECT_ROOT).as_posix() if uses_pca else None,
    "knn": {
        "n_neighbors": int(best["k"]), "weights": str(best["weights"]),
        "metric": str(best["metric"]), "algorithm": "brute",
    },
    "dev_metrics": {
        "macro_f1": float(best["macro_f1"]), "accuracy": float(best["accuracy"]),
        "mae_vnd": float(best["mae_vnd"]),
    },
    "training_split": "train", "refit_train_plus_dev": False,
    "selection_rule": "max unrounded Macro-F1, min unrounded MAE, feature order, k, weights order",
    "grid_configurations": int(len(all_results)),
    "test_history": "Previously evaluated and inspected in earlier experiments; not an untouched holdout.",
    "test_evaluation_in_this_notebook": False,
    "sha256": fingerprints,
    "environment_at_lock": {
        "python": platform.python_version(), "sklearn": sklearn.__version__,
        "opencv": cv2.__version__, "numpy": np.__version__, "pandas": pd.__version__,
    },
}

lock_file = output_dir / "locked_knn_config.json"
if lock_file.exists():
    print("Đã có cấu hình khóa; giữ nguyên file cũ:", lock_file)
    saved_lock = json.loads(lock_file.read_text(encoding="utf-8"))
    print("Cấu hình đang khóa:", saved_lock["feature_id"], saved_lock["knn"])
    print("Đầu vào hiện tại khớp lúc khóa:", saved_lock["sha256"] == fingerprints)
else:
    lock_file.write_text(json.dumps(locked_config, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Đã ghi cấu hình khóa:", lock_file)
    print("Cấu hình:", locked_config["feature_id"], locked_config["knn"])

Đã ghi cấu hình khóa: C:\CS231\src\KNN\results_224\selection\locked_knn_config.json
Cấu hình: HSV12 {'n_neighbors': 1, 'weights': 'uniform', 'metric': 'euclidean', 'algorithm': 'brute'}


## Sau bước này

Đọc cấu hình đã khóa và kiểm tra nhận xét của bạn có khớp bảng dev. Bước tiếp theo mới tạo notebook test, dùng đúng feature/pipeline và KNN trong file khóa. Không fit lại PCA hoặc scaler trên test; không gộp train+dev.

Nếu chạy lại notebook này, file khóa cũ được giữ nguyên. Bảng so sánh có thể được tạo lại, nhưng không tự thay thế cấu hình đã khóa. Nếu kiểm tra đầu vào khớp lúc khóa trả False, dừng trước bước test để xác minh thay đổi. Không sửa cấu hình chỉ vì thấy lỗi test.